In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path("/Users/I768838/Desktop/Probabilistic-quadrature")))

# Probabilistic Numerical Integration via Bayesian Quadrature and Active Sampling

**Author:** Yoan Desislavov Baychev  

## 1. Introduction

Numerical integration is a fundamental problem in scientific computing, statistics, and machine learning. In many practical settings, one is interested in approximating an integral of a function that may be expensive to evaluate, analytically intractable, or only accessible through a black-box routine. This makes the design of accurate and sample-efficient integration methods especially important.

Classical deterministic methods, such as the rectangle rule, the trapezoidal rule, and Simpson's rule, are widely used when the integration domain is simple and the function is sufficiently regular. Monte Carlo methods, on the other hand, are particularly attractive in probabilistic settings and in higher dimensions, since they naturally interpret integration as expectation under a probability distribution. However, standard Monte Carlo estimators typically ignore structural assumptions about the integrand and may therefore require a large number of function evaluations in order to achieve high accuracy.

Bayesian Quadrature (BQ) offers a probabilistic perspective on numerical integration. Instead of treating the integrand as an arbitrary deterministic function, BQ models it as a random function, typically by placing a Gaussian process prior over it. Once function values have been observed at a set of input locations, this prior is updated to a posterior distribution, which in turn induces a posterior distribution over the value of the integral. As a result, Bayesian Quadrature provides not only a point estimate of the integral, but also a principled uncertainty estimate.

The main goal of this project is to study Bayesian Quadrature as a probabilistic numerical method and to compare it to classical deterministic rules and Monte Carlo estimators. In addition, an active sampling strategy is considered, where new evaluation points are selected adaptively in order to reduce uncertainty in the integral estimate as efficiently as possible.

The report is organized as follows. First, the integration problem is formulated in probabilistic terms. Then, the necessary mathematical background on expectations and Gaussian processes is introduced. After that, the baseline integration methods and the Bayesian Quadrature framework are presented, followed by an active extension of the method. Finally, the implementation is outlined and the methods are compared experimentally.

## 2. Problem Formulation

In this project, the main object of interest is the integral of a function with respect to a measure. More precisely, given a measurable function $f$ and a probability measure $\mu$ on a domain $\mathcal{X}$, we consider the quantity

$$
I_\mu(f) = \int_{\mathcal{X}} f(x)\, d\mu(x).
$$

This formulation is broad enough to include many important special cases. For example, if $\mu$ is the uniform probability measure on an interval or a bounded domain, then the integral above is directly related to the standard deterministic integral over that region. Likewise, if $\mu$ is a Gaussian measure, then the same formulation naturally describes Gaussian-weighted integration problems.

Adopting the measure-theoretic point of view is particularly convenient in the context of probabilistic numerics. It makes the connection to expectation explicit, it aligns naturally with Monte Carlo estimation, and it allows Bayesian Quadrature to be formulated in a unified way across different choices of integration measures.

### 2.1 Expectation as an Integral

If $X \sim \mu$ is a random variable with distribution $\mu$, then the integral above can be written as an expectation:

$$
\mathbb{E}[f(X)] = \int_{\mathcal{X}} f(x)\, d\mu(x).
$$

This identity provides the conceptual bridge between numerical integration and probabilistic estimation. In particular, Monte Carlo methods approximate the target integral by drawing samples from $\mu$ and averaging the corresponding function values. Bayesian Quadrature builds on the same formulation, but additionally introduces a probabilistic model for the unknown function $f$.

When the integration domain is a bounded interval and the underlying measure is uniform, the relation to the standard Riemann integral is immediate. For example, if $X \sim \mathrm{Uniform}(a,b)$, then

$$
\mathbb{E}[f(X)] = \frac{1}{b-a}\int_a^b f(x)\, dx,
$$

which implies

$$
\int_a^b f(x)\, dx = (b-a)\,\mathbb{E}[f(X)].
$$

This observation shows that deterministic integration and probabilistic integration are not separate problems, but rather two closely related perspectives on the same task.

## 3. Mathematical Background

### 3.1 Random Variables and Expectation

Before defining a random variable formally, it is useful to introduce the probabilistic framework in which it is defined.

A **random experiment** is an experiment whose outcome cannot be predicted with certainty in advance. Each possible outcome of such an experiment is called an **elementary outcome**. The set of all elementary outcomes is denoted by $\Omega$ and is called the **sample space**.

An **event** is any subset $A \subseteq \Omega$. However, in probability theory, not every subset is necessarily assigned a probability directly. Instead, one works with a distinguished collection of subsets, denoted by $\mathcal{A}$, called a **$\sigma$-algebra**.

**Definition ($\sigma$-algebra).** Let $\Omega$ be a set and let $\mathcal{A}$ be a collection of subsets of $\Omega$. Then $\mathcal{A}$ is called a **$\sigma$-algebra** if:

- $\emptyset \in \mathcal{A}$,
- if $A \in \mathcal{A}$, then $A^c \in \mathcal{A}$,
- if $A_1, A_2, \dots \in \mathcal{A}$, then $\bigcup_{i=1}^\infty A_i \in \mathcal{A}$.

The pair $(\Omega, \mathcal{A})$ specifies which events are measurable. Once a probability measure $\mathbb{P}$ is defined on $\mathcal{A}$, the triple

$$
(\Omega, \mathcal{A}, \mathbb{P})
$$

is called a **probability space**.

**Definition (Random Variable).** Let $(\Omega, \mathcal{A}, \mathbb{P})$ be a probability space. A mapping

$$
X : \Omega \to \mathbb{R}
$$

is called a **random variable** if it is measurable, that is, if for every interval $(a,b) \subseteq \mathbb{R}$,

$$
X^{-1}((a,b)) \in \mathcal{A}.
$$

Equivalently, for every Borel set $B \subseteq \mathbb{R}$,

$$
X^{-1}(B) = \{\omega \in \Omega \mid X(\omega) \in B\} \in \mathcal{A}.
$$

In particular,

$$
X^{-1}((a,b)) = \{\omega \in \Omega \mid a < X(\omega) < b\} = \{a < X < b\} \in \mathcal{A}.
$$

A random variable is a measurable mapping from an underlying probability space to a numerical space, typically $\mathbb{R}$ or $\mathbb{R}^d$. In practice, random variables are used to describe uncertainty in the input to a function, and their distributions determine how expectations and probabilistic integrals are defined.

For a real-valued random variable $X$ with distribution $\mu$, the expectation of a function $f(X)$ is given by

$$
\mathbb{E}[f(X)] = \int f(x)\, d\mu(x),
$$

provided the integral is well-defined. This quantity is central to both Monte Carlo integration and Bayesian Quadrature. In the former, it is approximated by sample averages. In the latter, it is inferred through a probabilistic model of the integrand itself.

The expectation-based formulation is especially useful because it separates the role of the function from the role of the measure. The function $f$ describes what is being integrated, while the measure $\mu$ describes where the mass of the integration problem is concentrated. This distinction becomes important when comparing deterministic rules, Monte Carlo estimators, and kernel-based probabilistic methods.

### 3.2 Gaussian Processes

A Gaussian process (GP) is a collection of random variables indexed by the input variable $x \in \mathcal{X}$, such that every finite subset follows a joint Gaussian distribution. A Gaussian process is fully specified by its mean function $m(x)$ and covariance function $k(x,x')$, and is written as

$$
f \sim \mathcal{GP}(m(x), k(x,x')).
$$

Gaussian processes are widely used as nonparametric priors over functions. Instead of assuming a fixed parametric form for the integrand, a GP expresses prior beliefs about its regularity, smoothness, and correlation structure. After observing function values at a finite set of input locations, the prior is updated to a posterior distribution that reflects both the data and the prior assumptions encoded in the kernel.

In the context of Bayesian Quadrature, the Gaussian process serves as a probabilistic model for the unknown integrand. Since integration is a linear operator, a GP prior over functions induces a Gaussian distribution over the integral itself. This fact is what makes Bayesian Quadrature both analytically tractable and conceptually appealing.

### 3.3 Kernels and Prior Assumptions

The covariance function, or kernel, plays a central role in Gaussian process modeling. It determines how strongly function values at different input locations are correlated, and therefore encodes prior assumptions about the shape and regularity of the integrand.

In this project, two commonly used kernels are considered: the Radial Basis Function (RBF) kernel and the Matérn $3/2$ kernel. The RBF kernel assumes a very smooth latent function and is often appropriate when the integrand is believed to vary smoothly across the domain. The Matérn $3/2$ kernel is less restrictive and allows for rougher behavior, which may be more suitable when the function is only moderately smooth.

The choice of kernel is important because Bayesian Quadrature relies on the prior model to extrapolate from a limited number of function evaluations. If the kernel reflects the true structure of the integrand reasonably well, the resulting integral estimate can be highly accurate even with a small evaluation budget. Conversely, a poor kernel choice may lead to overconfidence or biased estimates.

## 4. Baseline Numerical Integration Methods

Before introducing Bayesian Quadrature, it is useful to review the baseline methods used for comparison in this project. These baselines include classical deterministic rules, Gaussian quadrature methods, and Monte Carlo estimators. Together, they provide a useful reference point for evaluating the strengths and limitations of probabilistic integration.

### 4.1 Classical Deterministic Rules

Classical deterministic integration rules approximate an integral by evaluating the function on a structured grid and combining those evaluations with predefined weights. These methods are especially effective in one-dimensional settings when the function is sufficiently smooth and the domain is simple.

The rectangle rule approximates the integral by summing the areas of rectangles whose heights are determined by the function values at selected grid points. The trapezoidal rule improves upon this by approximating the function linearly between neighboring nodes, thereby forming trapezoids rather than rectangles. Simpson's rule uses local quadratic interpolation and often achieves higher accuracy for smooth functions when the number of grid intervals is appropriate.

These methods are simple, interpretable, and computationally inexpensive. However, they are less flexible when the integration measure is non-uniform, and they do not naturally provide uncertainty estimates.

### 4.2 Gaussian Quadrature Rules

Gaussian quadrature methods are designed to achieve high accuracy by selecting both nodes and weights in an optimal way for a given class of weighted integration problems. Unlike uniform-grid rules, these methods adapt the quadrature nodes to the geometry of the weight function and can integrate polynomials of relatively high degree exactly.

Several Gaussian quadrature families are relevant in this context. Gauss-Legendre quadrature is suited for integration over bounded intervals with constant weight. Gauss-Hermite quadrature is designed for integrals with Gaussian-type weights over the real line. Gauss-Laguerre quadrature applies to exponentially weighted integrals over semi-infinite intervals, while Gauss-Chebyshev quadrature is specialized for Chebyshev-type weight functions on bounded domains.

These methods can be extremely effective when the target integral matches the weight structure for which the rule was derived. At the same time, this dependence on the underlying weight function means that Gaussian quadrature rules are not always directly comparable unless the integration problem has been formulated consistently.

### 4.3 Monte Carlo Estimators

Monte Carlo methods approximate integrals through random sampling. If $X_1, \dots, X_n$ are sampled independently from the target measure $\mu$, then the integral

$$
I_\mu(f) = \mathbb{E}[f(X)]
$$

can be approximated by the empirical average

$$
\hat{I}_n = \frac{1}{n}\sum_{i=1}^n f(X_i).
$$

The main strength of Monte Carlo integration lies in its generality. It applies naturally to integration with respect to probability measures, extends easily to higher dimensions, and does not require regular grids or analytically derived quadrature nodes. Its convergence properties are also largely dimension-independent, which makes it especially attractive in settings where deterministic methods become inefficient.

In this project, Monte Carlo estimators serve as an important probabilistic baseline. While they typically do not exploit smoothness assumptions about the integrand, they provide a simple and widely used reference point against which Bayesian Quadrature can be evaluated.

## 5. Bayesian Quadrature

Bayesian Quadrature approaches numerical integration by modeling the unknown integrand as a random function rather than as an arbitrary deterministic object. This allows prior knowledge about smoothness and correlation structure to be incorporated directly into the estimation procedure.

### 5.1 Probabilistic Model of the Integrand

Let $f$ denote the integrand of interest, and suppose that prior beliefs about $f$ are represented by a Gaussian process

$$
f \sim \mathcal{GP}(m(x), k(x,x')).
$$

Assume that the function is evaluated at points $x_1, \dots, x_n$, yielding observations $y_i = f(x_i)$. The observed dataset can then be written as

$$
D_n = \{(x_i, y_i)\}_{i=1}^n.
$$

Conditioning the Gaussian process prior on these observations produces a posterior Gaussian process. This posterior provides a data-informed probabilistic description of the function over the entire input domain, including both a posterior mean and a posterior covariance.

The posterior mean can be interpreted as the current best estimate of the integrand, while the posterior covariance quantifies residual uncertainty at unobserved points. Since the integral is a linear functional of the function, the same posterior model induces a probability distribution over the integral itself.

### 5.2 Posterior Distribution Over the Integral

A key property of Gaussian processes is that linear transformations of Gaussian random variables remain Gaussian. Therefore, if $f \mid D_n$ is a posterior Gaussian process, then the integral

$$
I_\mu(f) = \int f(x)\, d\mu(x)
$$

also has a Gaussian posterior distribution.

This posterior distribution is characterized by a mean and a variance. The posterior mean serves as the Bayesian Quadrature estimate of the integral, while the posterior variance quantifies uncertainty about the estimate under the assumed prior model. In other words, Bayesian Quadrature does not merely output a single numerical approximation; it also reports how uncertain that approximation is, given the available evaluations and the kernel assumptions.

This uncertainty-aware perspective is one of the main conceptual advantages of probabilistic numerics. It is particularly relevant when function evaluations are expensive and only a small number of observations can be collected.

### 5.3 Practical Computation of Kernel Integrals

In order to compute the posterior mean and variance of the integral, Bayesian Quadrature requires certain kernel-dependent quantities, such as the kernel mean and the integrated kernel variance. These involve expressions of the form

$$
\int k(x, x')\, d\mu(x)
\quad \text{and} \quad
\iint k(x, x')\, d\mu(x)\, d\mu(x').
$$

For some combinations of kernels and measures, these quantities admit closed-form expressions. In other cases, they must be approximated numerically. This introduces an additional practical layer to the implementation of Bayesian Quadrature: although the method is conceptually probabilistic, part of its internal machinery may still rely on numerical approximations.

This distinction is important when interpreting the results. The quality of a Bayesian Quadrature estimate depends not only on the observed function values and the chosen kernel, but also on how accurately these kernel-integral terms are computed under the chosen measure.

## 6. Active Bayesian Quadrature

In many applications, the evaluation budget is severely limited, which makes the placement of sampling points an important part of the integration problem. Rather than fixing all evaluation locations in advance, one may choose them sequentially in a data-dependent way.

### 6.1 Motivation

The motivation behind active Bayesian Quadrature is straightforward: if function evaluations are expensive, then each new evaluation should ideally be placed where it is expected to provide the greatest benefit. Since Bayesian Quadrature maintains a posterior distribution over the integral, it is natural to use this posterior to guide the selection of future evaluation points.

An active strategy aims to reduce uncertainty about the target integral as efficiently as possible. Instead of sampling blindly or relying on a fixed grid, the method uses the current probabilistic model to decide where additional information is most valuable.

### 6.2 Greedy Point Selection

A practical way to implement active Bayesian Quadrature is to evaluate a finite candidate set and greedily select the point that is expected to reduce the posterior uncertainty the most. After each newly selected point is evaluated, the Gaussian process model is updated and the selection criterion is recomputed.

This sequential procedure has two appealing properties. First, it is adaptive: the next point depends on all previous observations. Second, it is interpretable: each new evaluation is chosen because it is expected to improve the estimate of the integral in a measurable way.

Although greedy selection does not guarantee global optimality, it often performs well in practice and is computationally easier to implement than more sophisticated design strategies.

### 6.3 Expected Variance Reduction

In this project, the active strategy is based on variance reduction. The guiding principle is to select the next evaluation point so that the posterior variance of the integral is reduced as much as possible. Since the posterior variance summarizes residual uncertainty about the target quantity, minimizing it is a natural objective.

This criterion is especially well aligned with the probabilistic interpretation of Bayesian Quadrature. Unlike deterministic adaptive quadrature methods, which typically focus on local function behavior alone, active Bayesian Quadrature uses the global posterior model to account for both correlation structure and the integration measure.

As a result, the selected points are not merely chosen in regions of high function variation, but rather in regions that are expected to be most informative for the integral itself.

## 7. Implementation Overview

This project is organized in a modular way in order to separate the main conceptual components of the numerical integration framework.

### 7.1 Project Structure

The implementation is divided into several main parts. The `functions` module contains the abstractions for integrands and domains. The `measures` and `random_variables` modules describe the probability distributions with respect to which integration is performed. The `kernels` module contains the covariance functions used in the Gaussian process prior. Finally, the `numeric_integration` module includes the deterministic, Monte Carlo, Bayesian Quadrature, and active Bayesian Quadrature implementations.

This structure makes the code easier to extend and test. New kernels, measures, or integration methods can be added without changing the entire project architecture, which is especially useful in an experimental setting.

### 7.2 Core Classes

Several classes play a central role in the implementation. The Bayesian Quadrature model is responsible for fitting the Gaussian process to observed data, computing posterior quantities, and producing a posterior distribution over the integral. The dataset abstraction stores evaluation points and corresponding function values in a structured form. The active selector is responsible for choosing new evaluation points based on a variance-reduction criterion.

Additional supporting classes describe kernels, measures, domains, and baseline integration routines. Together, these components form a coherent framework for comparing deterministic and probabilistic approaches to numerical integration.

### 7.3 Design Choices

A key design choice in the project is the separation between the integration measure and the integrand itself. This reflects the probabilistic formulation of the task and allows the same function to be integrated under different measures when needed. Another important choice is the use of kernel abstractions, which makes it straightforward to compare different Gaussian process priors.

The implementation also emphasizes extensibility. Deterministic rules, Monte Carlo estimators, and Bayesian methods are represented as separate components with consistent interfaces wherever possible. This makes experimental comparison easier and supports the gradual development of new features.

## 8. Experimental Setup

The experimental part of the project is designed to compare classical deterministic integration rules, Monte Carlo estimators, Bayesian Quadrature, and Active Bayesian Quadrature under a controlled evaluation budget.

### 8.1 Test Functions

The selected test functions are intended to represent different levels of difficulty for numerical integration. Some are smooth and well aligned with the assumptions typically encoded by Gaussian process kernels, while others are more oscillatory or localized, making them harder to approximate accurately from a small number of evaluations.

Using multiple test functions is important because no single method is uniformly optimal across all settings. The performance of Bayesian Quadrature, in particular, depends on how well the kernel prior matches the true structure of the integrand.

### 8.2 Measures and Domains

The experiments are performed on integration problems defined over [state the domain here, e.g. bounded intervals, Gaussian measures, or both]. This choice reflects the fact that different integration methods are naturally suited to different types of domains and weighting structures.

When the measure is uniform over a bounded interval, the probabilistic formulation can be directly related to the standard deterministic integral. When the measure is Gaussian, the integration problem becomes naturally probabilistic, and kernel-based methods can be studied under a non-uniform weighting scheme.

### 8.3 Evaluation Budget

A central aspect of the comparison is the evaluation budget, that is, the number of times the integrand may be queried. This budget is varied across experiments in order to study how the accuracy of each method changes as more information becomes available.

This setup is particularly relevant for Bayesian Quadrature, whose main advantage is expected to appear in low-data regimes where each function evaluation is valuable. Active Bayesian Quadrature is also naturally evaluated in this setting, since its purpose is to allocate a limited number of evaluations as efficiently as possible.

### 8.4 Compared Methods

The methods compared in this report include classical deterministic quadrature rules, Monte Carlo estimators, Bayesian Quadrature, and its active extension. The deterministic methods serve as traditional numerical baselines, while Monte Carlo provides a probabilistic but structure-agnostic reference.

Bayesian Quadrature is evaluated as a model-based alternative that incorporates prior assumptions about smoothness through the kernel. Active Bayesian Quadrature is studied in order to determine whether adaptive point selection improves sample efficiency relative to passive or pre-fixed designs.

### 8.5 Error Metrics

The main evaluation metric used in the experiments is the absolute integration error,

$$
|\hat{I} - I^\star|,
$$

where $\hat{I}$ denotes the estimated integral and $I^\star$ denotes the reference or ground-truth value. This metric directly measures numerical accuracy and allows different methods to be compared on equal footing.

For Bayesian Quadrature, the posterior standard deviation is also of interest, since it reflects the method's internal uncertainty assessment. Comparing this uncertainty estimate to the actual error can provide useful insight into how informative the posterior distribution is in practice.